In [5]:
# 检测图像阈值，MetaCloak模式
from torchvision import transforms
import torch
from pathlib import Path
from PIL import Image

def load_data(data_dir, size=512, center_crop=True) -> torch.Tensor:
    image_transforms = transforms.Compose(
        [
            transforms.Resize(size, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(size) if center_crop else transforms.RandomCrop(size),
            transforms.ToTensor(),
            # transforms.Normalize([0.5], [0.5]),
        ]
    )

    images = [image_transforms(Image.open(i).convert("RGB")) for i in sorted(list(Path(data_dir).iterdir()))]
    images = torch.stack(images)
    return images

weight_type = torch.bfloat16
clean_leaf_id_pixel_values = load_data('/data1/humw/Datasets/VGGFace2/n000050/set_B').to(dtype=weight_type)
adv_leaf_id_pixel_values = load_data('/data1/humw/Codes/My-Anti-DreamBooth/outputs/adversarial_images/Encoder_attack_conda-photomaker_test_vae15-ipadapter-photomaker_mix_eot-0_yingbu_agm-2_norm-0/n000050').to(dtype=weight_type)
et = adv_leaf_id_pixel_values - clean_leaf_id_pixel_values

et = abs(et)
print("16/255:{}".format(16/255))
print("et min:{}".format(et.min()))
print("et max:{}".format(et.max()))
print("et mean:{}".format(et.mean()))

et = et.reshape(-1)
a = et
cnt1 = 0
cnt2 = 0
for t in a:
    if t > 16/255:
        cnt1 = cnt1 + 1
    if t > 17/255:
        cnt2 = cnt2 + 1
print("proportion of pixels larger than 16/255:{}".format(cnt1/et.shape[0]))
print("proportion of pixels larger than 17/255:{}".format(cnt2/et.shape[0]))

16/255:0.06274509803921569
et min:0.0
et max:0.068359375
et mean:0.0439453125


KeyboardInterrupt: 

In [ ]:
clean_leaf_id_pixel_values.min()

In [ ]:
clean_leaf_id_pixel_values.max()

In [ ]:
clean_leaf_id_pixel_values.mean()

In [ ]:
et = adv_leaf_id_pixel_values - clean_leaf_id_pixel_values
print(et.min())
print(et.max())

In [ ]:
import os

print(sorted(os.listdir("/data1/humw/Codes/My-Anti-DreamBooth/outputs/customization_outputs/ASPL")))

In [ ]:
import random
import numpy as np
seed = 1
random.seed(seed) # python的随机种子一样

In [ ]:
a = random.randint(0, 10)
print(a)

In [ ]:
import os

dir_path = "/data1/humw/Codes/My-Anti-DreamBooth/outputs/customization_outputs/ASPL_ace-plus_VGGFace2_SD15_mist"

for dir in os.listdir(dir_path):
    if dir.startswith("000"):
        print(dir)
        target_dir =  os.path.join(dir_path, dir)
        os.rmdir(target_dir)

In [ ]:
import torch

# 定义张量 A 和 B
A = torch.randn(2, 3, requires_grad=True)
# B = torch.randn(2, 3, requires_grad=True)
B = torch.randn(2, 3, requires_grad=True)
print("A:{}".format(A))
print("B:{}".format(B))

# 拼接张量
C = torch.cat([A, B], dim=1)
print("C:{}".format(C))

# 对 C 进行操作并计算梯度
loss = C.sum()  # 一个简单的损失函数
loss.backward()

# 查看梯度
print("A 的梯度:", A.grad)
print("B 的梯度:", B.grad)

## 自定义数据变换

In [ ]:
import torchvision.transforms as transforms
import random

def get_length(length, num_block=2):
    rand = np.random.uniform(2, size=num_block)
    rand_norm = np.round(rand/rand.sum()*length).astype(np.int32)
    rand_norm[rand_norm.argmax()] += length - rand_norm.sum()
    return tuple(rand_norm)

def shuffle_single_dim(x, dim):
    lengths = get_length(x.size(dim))
    x_strips = list(x.split(lengths, dim=dim))
    random.shuffle(x_strips)
    return x_strips

def image_rotation(x):
    rotation_transform = transforms.RandomRotation(degrees=(-24, 24), interpolation=transforms.InterpolationMode.BILINEAR)
    return  rotation_transform(x)

def shuffle(x):
    dims = [2,3]
    random.shuffle(dims)
    x_strips = shuffle_single_dim(x, dims[0])
    return torch.cat([torch.cat(shuffle_single_dim(image_rotation(x_strip), dim=dims[1]), dim=dims[1]) for x_strip in x_strips], dim=dims[0])
    # return torch.cat([torch.cat(shuffle_single_dim(x_strip, dim=dims[1]), dim=dims[1]) for x_strip in x_strips], dim=dims[0])


def bsr_transform(x):
    """
    Scale the input for BSR
    """
    # return torch.cat([shuffle(x) for _ in range(num_scale)])
    return shuffle(x)

In [ ]:
# 定义变换序列
transform_pipeline = transforms.Compose([
    transforms.Resize((512, 512)),  # 调整图像大小
    bsr_transform, # 应用自定义预处理
    # transforms.ToTensor(),          # 将图像转换为Tensor
    # transforms.Normalize(mean=[0.5], std=[0.5])  # 归一化
])

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torch
from pathlib import Path
from PIL import Image
import numpy as np
import os


def load_data(data_dir="", size=512, center_crop=True) -> torch.Tensor:
    def image_to_numpy(image):
        return np.array(image).astype(np.uint8)
    # more robust loading to avoid loaing non-image files
    images = [] 
    for i in list(Path(data_dir).iterdir()):
        if not i.suffix in [".jpg", ".png", ".jpeg"]:
            continue
        else:
            images.append(image_to_numpy(Image.open(i).convert("RGB")))
    images = [Image.fromarray(i).resize((size, size), 2) for i in images]
    images = np.stack(images)
    # from B x H x W x C to B x C x H x W
    images = torch.from_numpy(images).permute(0, 3, 1, 2).float()
    assert images.shape[-1] == images.shape[-2]
    return images

def save_image(save_dir, input_dir, perturbed_data):
    os.makedirs(save_dir, exist_ok=True)
    noised_imgs = perturbed_data.detach()
    img_names = [
        str(instance_path).split("/")[-1]
        for instance_path in list(Path(input_dir).iterdir())
    ]
    for img_pixel, img_name in zip(noised_imgs, img_names):
        save_path = os.path.join(save_dir, img_name)
        Image.fromarray(
            img_pixel.clamp(0, 255).to(torch.uint8).permute(1, 2, 0).cpu().numpy()
        ).save(save_path)
    print("save images to {}".format(save_dir))
    
images = load_data("/data1/humw/Codes/My-Anti-DreamBooth/data/n000050/set_B")
transformed_images = transform_pipeline(images)
save_image(save_dir="/data1/humw/Codes/My-Anti-DreamBooth/data/tranformed_images", input_dir="/data1/humw/Codes/My-Anti-DreamBooth/data/n000050/set_B", perturbed_data=transformed_images)

In [6]:
# Manually extract values from the new image for Ensemble (TED)
# Based on the table in the image, the values under each method are:

# Recalculate averages
fdf_values = [0.460, 0.871, 0.009, 0.208]
ism_values = [0.136, 0.069, 0.147, 0.144]

# Calculate averages
average_fdf = round(sum(fdf_values) / len(fdf_values), 3)
average_ism = round(sum(ism_values) / len(ism_values), 3)

average_fdf, average_ism


(0.387, 0.124)